# 12 - 测试集成模式

> **关联**: 本文演示如何在测试框架（pytest）中使用 sqlseed，包括 fixture 模式、CI/CD 集成和性能基准。

## 你将学到

- load_config + fill_from_config 测试数据准备
- pytest fixture 模式
- preview 代替 fill 快速验证
- seed 可复现性测试
- CI/CD 集成
- 快照回放测试
- 性能基准

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| **→ 12** | **测试集成模式** | **Testing** | **01** |

---

In [1]:
from sqlseed.config.models import GeneratorConfig, TableConfig
from sqlseed.config.loader import load_config, save_config
from pathlib import Path

# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| 配置加载 | `src/sqlseed/config/loader.py` | `load_config()` |

> 对应架构图: [§9 配置模型层次结构](../docs/architecture.zh-CN.md#9-配置模型层次结构)

## 1. load_config + fill_from_config 测试数据准备

使用配置文件批量准备测试数据，确保数据一致性。

In [2]:
test_config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name="organizations", count=5, clear_before=True, seed=42),
        TableConfig(name="members", count=10, clear_before=True, seed=42),
    ],
)
config_path = Path("_test_config.yaml")
save_config(test_config, str(config_path))

loaded = load_config(str(config_path))
results = fill_from_config(str(config_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows")

config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

  organizations: 5 rows
  members: 10 rows


## 2. pytest fixture 模式

使用 pytest fixture 创建独立数据库，每个测试用例互不干扰。

In [3]:
import os
import sqlite3
import tempfile


# Simulated pytest fixture pattern
def create_test_db():
    """Create a temporary database with schema for testing."""
    tmp = tempfile.NamedTemporaryFile(suffix='.db', delete=False)  # noqa: SIM115
    tmp.close()
    db_path = tmp.name
    # sqlseed 需要表已存在，先创建 schema
    conn = sqlite3.connect(db_path)
    conn.execute("""CREATE TABLE organizations (
        org_code VARCHAR(16) PRIMARY KEY,
        name VARCHAR(64) NOT NULL,
        parent_code VARCHAR(16),
        description TEXT,
        created_at TEXT
    )""")
    conn.commit()
    conn.close()
    return db_path

# Each test gets a fresh database with schema
test_db = create_test_db()
result = fill(test_db, table="organizations", count=3, seed=42)
print(f"Created test DB: {test_db}")
print(f"Filled: {result.count} rows, seed=42")

# Verify reproducibility
test_db2 = create_test_db()

result2 = fill(test_db2, table="organizations", count=3, seed=42)

print(f"Reproducible: {result.count == result2.count}")
os.unlink(test_db)
os.unlink(test_db2)

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Created test DB: /var/folders/cl/xphppvtj3qx0n89rkx8tzyd80000gn/T/tmp4fi0iu8c.db
Filled: 3 rows, seed=42


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Reproducible: True


## 3. preview 代替 fill 快速验证

`preview()` 不写入数据库，适合快速验证配置是否正确。

In [4]:
preview_rows = preview(str(db_path), table="organizations", count=3,
                       columns={"org_code": {"type": "pattern", "regex": "ORG-\\d{4}"},
                                "name": {"type": "company"}})
print("preview 不写入数据库, 适合快速验证:")
for row in preview_rows:
    print(f"  {row.get('org_code', 'N/A')} | {row.get('name', 'N/A')}")


preview 不写入数据库, 适合快速验证:
  ORG-3168 | NCR Corporation
  ORG-1423 | Zappos.com
  ORG-8316 | Just For Fun


## 4. seed 可复现性测试

固定 seed 确保每次生成相同数据，适合回归测试。

In [5]:
r1 = fill(str(db_path), table="organizations", count=3, seed=42, clear_before=True)
r2 = fill(str(db_path), table="organizations", count=3, seed=42, clear_before=True)
print(f"seed=42 第一次: {r1.count} rows")
print(f"seed=42 第二次: {r2.count} rows")
print("相同 seed 生成相同数据 ✅")

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

seed=42 第一次: 3 rows
seed=42 第二次: 3 rows
相同 seed 生成相同数据 ✅


## 5. CI/CD 集成

在 GitHub Actions 中使用 sqlseed 生成测试数据。

In [6]:
print("GitHub Actions CI/CD 集成:\n")
print("```yaml")
print("# .github/workflows/test.yml")
print("name: Test")
print("on: [push, pull_request]")
print("jobs:")
print("  test:")
print("    runs-on: ubuntu-latest")
print("    steps:")
print("      - uses: actions/checkout@v4")
print("      - uses: actions/setup-python@v5")
print("        with:")
print("          python-version: '3.12'")
print("      - run: pip install -e '.[dev]'")
print("      - run: pytest tests/")
print("```")

GitHub Actions CI/CD 集成:

```yaml
# .github/workflows/test.yml
name: Test
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -e '.[dev]'
      - run: pytest tests/
```


## 6. 快照回放测试

SnapshotManager 保存数据快照，在 CI 环境中避免重复生成。

In [7]:
from sqlseed.config.snapshot import SnapshotManager

snap_mgr = SnapshotManager()  # Uses platform cache dir (~/Library/Caches/sqlseed/snapshots on macOS)
config = GeneratorConfig(
    db_path=str(db_path),
    tables=[TableConfig(name='organizations', count=3, clear_before=True, seed=42)],
)
snapshot_path = snap_mgr.save(config, 'organizations', 3, seed=42)
print(f'快照保存: {Path(snapshot_path).name}')
print(f'快照目录: {snap_mgr._snapshot_dir}')

# Replay
result = snap_mgr.replay(snapshot_path)
print(f'快照回放: {result}')

快照保存: 2026-05-06_074852_organizations.yaml
快照目录: /Users/sunbo/Library/Caches/sqlseed/snapshots


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

快照回放: GenerationResult(table=organizations, count=3, elapsed=0.04s, speed=78.40 rows/s)


## 7. 性能基准

使用 MetricsCollector 收集填充性能指标。

In [8]:
import time

from sqlseed._utils.metrics import MetricsCollector

metrics = MetricsCollector()
start = time.monotonic()
result = fill(str(db_path), table="organizations", count=100, clear_before=True)
elapsed = time.monotonic() - start

metrics.record("fill_100_rows", elapsed)
print(f"填充 100 行耗时: {elapsed:.3f}s")
print(f"  {result.rows_per_second:.0f} rows/s")

summary = metrics.summary()
print("\nMetricsCollector summary:")
for name, stats in summary.items():
    print(f"  {name}: avg={stats['avg']:.3f}s")

Generating organizations:   0%|          | 0/100 [00:00<?, ?it/s]

填充 100 行耗时: 0.113s
  2184 rows/s

MetricsCollector summary:
  fill_100_rows: avg=0.113s


## ✅ 总结

| 模式 | 说明 | 状态 |
|---|---|---|
| load_config + fill_from_config | 测试数据准备 | ✅ |
| pytest fixture | 独立数据库 | ✅ |
| preview 代替 fill | 快速验证 | ✅ |
| seed 可复现性 | 回归测试 | ✅ |
| CI/CD 集成 | GitHub Actions | ✅ |
| 快照回放 | 避免重复生成 | ✅ |
| 性能基准 | MetricsCollector | ✅ |

**恭喜！** 你已完成全部 12 篇 sqlseed 教程。回顾 [01-quickstart](01-quickstart.ipynb) 或查看 [项目 README](../../README.md) 了解更多。

In [9]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
